In [1]:
import requests
import xarray as xr
import pandas as pd
from pathlib import Path
import rasterio
import numpy as np
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import requests
import re
import os

from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
# ============================================================
# NOAA GEFSv12 REFORECAST DOWNLOADER
# WITH PARALLEL DOWNLOADS + MONTHLY PROCESSED CACHE
# ============================================================


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

START_DATE = "2015-01-01"
END_DATE = "2016-01-1"

# Forecast member:
# c00 = control forecast
MEMBER = "c00"

# Variables from NOAA GEFS reforecast
VARIABLES = {
    "tmp_2m": "predicted_temp_k",   # 2 metre forecast temperature, Kelvin
    "apcp_sfc": "precipitation",    # accumulated precipitation
    "tcdc_eatm": "cloud_cover"      # total cloud cover
}

# Forecast lead times to extract
LEAD_HOURS = [30]

# Parallel downloads
# Start with 4. If stable, try 6 or 8.
MAX_WORKERS = 4

# Local folder for cached raw GRIB2 files
DATA_DIR = Path("gefs_cache")
DATA_DIR.mkdir(exist_ok=True)

# Local folder for monthly processed forecast CSV files
MONTHLY_CACHE_DIR = Path("gefs_monthly_cache")
MONTHLY_CACHE_DIR.mkdir(exist_ok=True)

# Request settings
REQUEST_TIMEOUT = 180
CHUNK_SIZE = 1024 * 1024  # 1 MB chunks

# If True, re-process monthly CSV even if it already exists
FORCE_REBUILD_MONTHLY_CACHE = False


# ------------------------------------------------------------
# Build NOAA URL and local filename
# ------------------------------------------------------------

def get_gefs_url_and_filename(year, run, variable, member="c00"):
    """
    Builds the remote NOAA GEFS URL and local cached filename.
    """

    url = (
        f"https://noaa-gefs-retrospective.s3.amazonaws.com/"
        f"GEFSv12/reforecast/{year}/{run}/{member}/Days:1-10/"
        f"{variable}_{run}_{member}.grib2"
    )

    folder = DATA_DIR / str(year) / run / member
    folder.mkdir(parents=True, exist_ok=True)

    filename = folder / f"{variable}_{run}_{member}.grib2"

    return url, filename


# ------------------------------------------------------------
# Download one NOAA GEFS file with cache
# ------------------------------------------------------------

def download_gefs_file(year, run, variable, member="c00"):
    """
    Downloads one GEFSv12 reforecast GRIB2 file only if it does
    not already exist locally.

    Returns:
        Path to local GRIB2 file, or None if the file is missing.
    """

    url, filename = get_gefs_url_and_filename(
        year=year,
        run=run,
        variable=variable,
        member=member
    )

    # --------------------------------------------------------
    # Use cached file if already downloaded
    # --------------------------------------------------------
    if filename.exists() and filename.stat().st_size > 0:
        return filename

    temp_filename = filename.with_suffix(".grib2.tmp")

    # If old failed temp file exists, remove it
    if temp_filename.exists():
        try:
            temp_filename.unlink()
        except OSError:
            pass

    # --------------------------------------------------------
    # Download file
    # --------------------------------------------------------
    try:
        response = requests.get(
            url,
            stream=True,
            timeout=REQUEST_TIMEOUT
        )

        if response.status_code != 200:
            print(f"Missing/unavailable: HTTP {response.status_code} {url}")
            return None

        with open(temp_filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

        # Only rename after successful complete download
        temp_filename.rename(filename)

        print("Downloaded:", filename)
        return filename

    except requests.RequestException as e:
        print("Request failed:", url)
        print(e)
        return None

    except OSError as e:
        print("File write failed:", filename)
        print(e)
        return None


# ------------------------------------------------------------
# Create list of files needed for a date range
# ------------------------------------------------------------

def make_download_tasks(start_date, end_date):
    """
    Creates all required download jobs.

    One job = one date/run/variable file.
    """

    dates = pd.date_range(start_date, end_date, freq="D")
    tasks = []

    for date in dates:
        year = date.year
        run = date.strftime("%Y%m%d") + "00"

        for variable in VARIABLES.keys():
            tasks.append(
                {
                    "year": year,
                    "run": run,
                    "variable": variable,
                    "member": MEMBER
                }
            )

    return tasks


# ------------------------------------------------------------
# Download many GEFS files in parallel
# ------------------------------------------------------------

def download_files_parallel(tasks, max_workers=4):
    """
    Downloads all required raw GRIB2 files in parallel.

    Uses local cache, so already-downloaded files are skipped.
    """

    print()
    print("Starting parallel download")
    print("--------------------------")
    print("Files needed:", len(tasks))
    print("Workers:", max_workers)
    print()

    downloaded = []
    missing = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        future_to_task = {
            executor.submit(
                download_gefs_file,
                task["year"],
                task["run"],
                task["variable"],
                task["member"]
            ): task
            for task in tasks
        }

        completed = 0

        for future in as_completed(future_to_task):

            task = future_to_task[future]
            completed += 1

            try:
                filename = future.result()

                if filename is None:
                    missing.append(task)
                else:
                    downloaded.append(filename)

            except Exception as e:
                print("Unexpected download error:")
                print(task)
                print(e)
                missing.append(task)

            if completed % 20 == 0 or completed == len(tasks):
                print(f"Download progress: {completed}/{len(tasks)}")

    print()
    print("Parallel download finished")
    print("--------------------------")
    print("Available files:", len(downloaded))
    print("Missing files:", len(missing))
    print()

    return downloaded, missing


# ------------------------------------------------------------
# Read one GRIB2 variable into a dataframe
# ------------------------------------------------------------

def read_gefs_variable(filename, new_column_name):
    """
    Reads one GEFS GRIB2 file.

    Steps:
      1. Opens the file with xarray/cfgrib
      2. Reduces the 0.25-degree grid to about 1-degree spacing
      3. Keeps only the USA approximate box
      4. Keeps only selected lead times
      5. Returns a dataframe with a clear column name
    """

    try:
        ds = xr.open_dataset(
            filename,
            engine="cfgrib",
            backend_kwargs={
                # avoids creating lots of .idx files beside GRIBs
                "indexpath": ""
            }
        )

    except Exception as e:
        print("Could not open GRIB file:")
        print(filename)
        print(e)
        return None

    try:
        # GEFS grid is 0.25 degrees.
        # Taking every 4th point gives about 1-degree spacing.
        ds = ds.isel(
            latitude=slice(0, None, 4),
            longitude=slice(0, None, 4)
        )

        # USA approximate box.
        # GEFS latitude runs north -> south, so slice(50, 25) is correct.
        # GEFS longitude uses 0-360 degrees.
        # 235 to 295 = roughly western/central USA.
        ds = ds.sel(
            latitude=slice(50, 25),
            longitude=slice(235, 295)
        )

        rows = []

        for lead in LEAD_HOURS:

            try:
                ds_lead = ds.sel(step=pd.Timedelta(hours=lead))

            except KeyError:
                print(f"Lead time {lead}h not found in {filename}")
                continue

            df_var = ds_lead.to_dataframe().reset_index()
            df_var["lead_hours"] = lead
            rows.append(df_var)

        if not rows:
            return None

        df_var = pd.concat(rows, ignore_index=True)

        # Get internal GRIB variable name used by cfgrib/xarray.
        # Example: tmp_2m may appear as t2m.
        data_var_name = list(ds.data_vars)[0]

        df_var = df_var.rename(columns={data_var_name: new_column_name})

        keep_columns = [
            "time",
            "valid_time",
            "lead_hours",
            "latitude",
            "longitude",
            new_column_name
        ]

        df_var = df_var[keep_columns]

        return df_var

    finally:
        ds.close()


# ------------------------------------------------------------
# Process one day from local cached GRIB2 files
# ------------------------------------------------------------

def process_one_day(date):
    """
    Reads all cached variables for one day and merges them into
    one daily dataframe.

    Returns:
        df_day or None
    """

    year = date.year
    run = date.strftime("%Y%m%d") + "00"

    daily_dfs = []

    for variable, new_column_name in VARIABLES.items():

        _, filename = get_gefs_url_and_filename(
            year=year,
            run=run,
            variable=variable,
            member=MEMBER
        )

        if not filename.exists() or filename.stat().st_size == 0:
            print("Missing cached file, skipping:", filename)
            return None

        df_var = read_gefs_variable(
            filename=filename,
            new_column_name=new_column_name
        )

        if df_var is None:
            print("Could not read variable, skipping day:", run, variable)
            return None

        daily_dfs.append(df_var)

    if len(daily_dfs) != len(VARIABLES):
        print("Skipping day because not all variables were available:", run)
        return None

    df_day = daily_dfs[0]

    for next_df in daily_dfs[1:]:
        df_day = df_day.merge(
            next_df,
            on=[
                "time",
                "valid_time",
                "lead_hours",
                "latitude",
                "longitude"
            ],
            how="inner"
        )

    df_day["year"] = year
    df_day["run"] = run

    return df_day


# ------------------------------------------------------------
# Get list of monthly periods
# ------------------------------------------------------------

def get_month_periods(start_date, end_date):
    """
    Splits date range into monthly chunks.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    month_starts = pd.date_range(
        start=start.replace(day=1),
        end=end,
        freq="MS"
    )

    periods = []

    for month_start in month_starts:

        month_end = month_start + pd.offsets.MonthEnd(0)

        actual_start = max(start, month_start)
        actual_end = min(end, month_end)

        periods.append((actual_start, actual_end))

    return periods


# ------------------------------------------------------------
# Build one monthly forecast dataframe
# ------------------------------------------------------------

def build_monthly_forecast_dataframe(month_start, month_end):
    """
    Downloads all needed files for one month in parallel,
    then processes that month into a dataframe.
    """

    month_label = month_start.strftime("%Y_%m")
    monthly_cache_file = MONTHLY_CACHE_DIR / f"forecast_{month_label}.csv"

    if (
        monthly_cache_file.exists()
        and monthly_cache_file.stat().st_size > 0
        and not FORCE_REBUILD_MONTHLY_CACHE
    ):
        print()
        print("Using monthly processed cache:", monthly_cache_file)
        return pd.read_csv(
            monthly_cache_file,
            parse_dates=["time", "valid_time"]
        )

    print()
    print("============================================================")
    print("Processing month:", month_label)
    print("Date range:", month_start.date(), "to", month_end.date())
    print("============================================================")

    # --------------------------------------------------------
    # 1. Download all raw files for this month in parallel
    # --------------------------------------------------------
    tasks = make_download_tasks(month_start, month_end)

    downloaded, missing = download_files_parallel(
        tasks=tasks,
        max_workers=MAX_WORKERS
    )

    if missing:
        print("Some files were missing for month:", month_label)
        print("Missing count:", len(missing))

    # --------------------------------------------------------
    # 2. Process cached raw files day-by-day
    # --------------------------------------------------------
    all_days = []
    dates = pd.date_range(month_start, month_end, freq="D")

    for date in dates:
        run = date.strftime("%Y%m%d") + "00"
        print("Reading cached GRIB files for:", run)

        df_day = process_one_day(date)

        if df_day is not None:
            all_days.append(df_day)

    if not all_days:
        print("No usable days for month:", month_label)
        return None

    forecast_month_df = pd.concat(all_days, ignore_index=True)

    # Convert forecast temperature from Kelvin to Celsius
    forecast_month_df["predicted_temp_c"] = (
        forecast_month_df["predicted_temp_k"] - 273.15
    )

    # Drop Kelvin column after conversion
    forecast_month_df = forecast_month_df.drop(columns=["predicted_temp_k"])

    # Make sure time columns are datetime
    forecast_month_df["time"] = pd.to_datetime(forecast_month_df["time"])
    forecast_month_df["valid_time"] = pd.to_datetime(
        forecast_month_df["valid_time"]
    )

    # Clean column order
    forecast_month_df = forecast_month_df[
        [
            "year",
            "run",
            "time",
            "valid_time",
            "lead_hours",
            "latitude",
            "longitude",
            "predicted_temp_c",
            "precipitation",
            "cloud_cover"
        ]
    ]

    # --------------------------------------------------------
    # 3. Save monthly processed cache
    # --------------------------------------------------------
    temp_cache_file = monthly_cache_file.with_suffix(".csv.tmp")

    forecast_month_df.to_csv(temp_cache_file, index=False)
    temp_cache_file.rename(monthly_cache_file)

    print("Saved monthly processed cache:", monthly_cache_file)
    print("Monthly shape:", forecast_month_df.shape)

    return forecast_month_df


# ------------------------------------------------------------
# Build full forecast dataframe from monthly caches
# ------------------------------------------------------------

def build_forecast_dataframe():
    """
    Builds forecast_df using monthly cache files.

    For each month:
      1. download raw GRIB2 files in parallel
      2. read and process cached GRIB2 files
      3. save processed monthly CSV
      4. combine monthly CSVs
    """

    monthly_dfs = []

    periods = get_month_periods(START_DATE, END_DATE)

    for month_start, month_end in periods:

        df_month = build_monthly_forecast_dataframe(
            month_start=month_start,
            month_end=month_end
        )

        if df_month is not None:
            monthly_dfs.append(df_month)

    if not monthly_dfs:
        raise RuntimeError("No forecast data was downloaded or processed.")

    forecast_df = pd.concat(monthly_dfs, ignore_index=True)

    return forecast_df


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

forecast_df = build_forecast_dataframe()

print()
print("Forecast dataframe preview:")
print(forecast_df.head())

print()
print("Forecast dataframe shape:")
print(forecast_df.shape)


Processing month: 2015_01
Date range: 2015-01-01 to 2015-01-31

Starting parallel download
--------------------------
Files needed: 93
Workers: 4

Downloaded: gefs_cache\2015\2015010100\c00\apcp_sfc_2015010100_c00.grib2
Downloaded: gefs_cache\2015\2015010200\c00\tmp_2m_2015010200_c00.grib2
Downloaded: gefs_cache\2015\2015010100\c00\tcdc_eatm_2015010100_c00.grib2
Downloaded: gefs_cache\2015\2015010200\c00\tcdc_eatm_2015010200_c00.grib2
Downloaded: gefs_cache\2015\2015010100\c00\tmp_2m_2015010100_c00.grib2
Downloaded: gefs_cache\2015\2015010200\c00\apcp_sfc_2015010200_c00.grib2
Downloaded: gefs_cache\2015\2015010300\c00\apcp_sfc_2015010300_c00.grib2
Downloaded: gefs_cache\2015\2015010400\c00\apcp_sfc_2015010400_c00.grib2
Downloaded: gefs_cache\2015\2015010300\c00\tcdc_eatm_2015010300_c00.grib2
Downloaded: gefs_cache\2015\2015010400\c00\tcdc_eatm_2015010400_c00.grib2
Downloaded: gefs_cache\2015\2015010500\c00\apcp_sfc_2015010500_c00.grib2
Downloaded: gefs_cache\2015\2015010300\c00\tmp_2m

In [ ]:
# ============================================================
# NOAA/NCEP REANALYSIS 2 ACTUAL 2M TEMPERATURE WITH SAFE CACHE
#
# Downloads the yearly NOAA/NCEP Reanalysis 2 actual 2m air
# temperature NetCDF file once, stores it locally, and reuses it.
#
# Handles broken downloads using retries + resume.
# ============================================================

from pathlib import Path
import time
import requests
import pandas as pd
import xarray as xr


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

START_DATE = "2015-01-01"
END_DATE   = "2016-01-01"

YEAR = pd.to_datetime(START_DATE).year

ACTUAL_CACHE_DIR = Path("noaa_actual_cache")
ACTUAL_CACHE_DIR.mkdir(exist_ok=True)

REQUEST_TIMEOUT = 180
CHUNK_SIZE = 1024 * 1024      # 1 MB
MAX_RETRIES = 10
WAIT_SECONDS = 20


# ------------------------------------------------------------
# Helper: get remote file size
# ------------------------------------------------------------

def get_remote_file_size(url):
    """
    Returns the file size from the server using a HEAD request.
    If the server does not provide Content-Length, returns None.
    """

    try:
        r = requests.head(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)

        if r.status_code == 200:
            size = r.headers.get("Content-Length")
            if size is not None:
                return int(size)

    except requests.RequestException:
        pass

    return None


# ------------------------------------------------------------
# Download yearly actual temperature file safely
# ------------------------------------------------------------

def download_actual_year_file(year):
    """
    Downloads the yearly NOAA/NCEP Reanalysis 2 actual 2m
    temperature file if it does not already exist locally.

    Uses:
    - local cache
    - temporary file
    - retries
    - resume if partial download exists
    - size check

    Returns:
        Path to local NetCDF file.
    """

    local_file = ACTUAL_CACHE_DIR / f"air.2m.gauss.{year}.nc"
    temp_file = ACTUAL_CACHE_DIR / f"air.2m.gauss.{year}.nc.tmp"

    url = (
        "https://psl.noaa.gov/thredds/fileServer/"
        f"Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.{year}.nc"
    )

    print("NOAA actual file URL:")
    print(url)

    remote_size = get_remote_file_size(url)

    if remote_size is not None:
        print(f"Remote file size: {remote_size:,} bytes")

    # --------------------------------------------------------
    # Use cached file if complete
    # --------------------------------------------------------

    if local_file.exists() and local_file.stat().st_size > 0:
        local_size = local_file.stat().st_size

        if remote_size is None or local_size == remote_size:
            print("Using cached actual file:", local_file)
            print(f"Local file size: {local_size:,} bytes")
            return local_file

        else:
            print("Cached file size does not match remote file.")
            print(f"Local:  {local_size:,}")
            print(f"Remote: {remote_size:,}")
            print("Deleting bad cached file.")
            local_file.unlink()

    # --------------------------------------------------------
    # Retry download
    # --------------------------------------------------------

    for attempt in range(1, MAX_RETRIES + 1):

        print()
        print(f"Download attempt {attempt}/{MAX_RETRIES}")

        # Resume from temp file if it exists
        resume_from = 0

        if temp_file.exists():
            resume_from = temp_file.stat().st_size
            print(f"Found partial download: {resume_from:,} bytes")

        headers = {}

        if resume_from > 0:
            headers["Range"] = f"bytes={resume_from}-"
            print("Trying to resume download...")

        try:
            with requests.get(
                url,
                stream=True,
                timeout=REQUEST_TIMEOUT,
                headers=headers
            ) as response:

                # 200 = fresh download
                # 206 = resumed partial content
                if response.status_code not in (200, 206):
                    raise RuntimeError(
                        f"Download failed: HTTP {response.status_code}"
                    )

                # If server ignores Range and sends full file again,
                # restart temp file from zero.
                if resume_from > 0 and response.status_code == 200:
                    print("Server did not resume. Restarting download.")
                    resume_from = 0

                mode = "ab" if resume_from > 0 and response.status_code == 206 else "wb"

                downloaded_this_attempt = 0

                with open(temp_file, mode) as f:
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            downloaded_this_attempt += len(chunk)

                current_size = temp_file.stat().st_size

                print(f"Downloaded this attempt: {downloaded_this_attempt:,} bytes")
                print(f"Temporary file size:     {current_size:,} bytes")

                # ------------------------------------------------
                # Check whether download is complete
                # ------------------------------------------------

                if remote_size is not None:
                    if current_size == remote_size:
                        temp_file.rename(local_file)
                        print("Download complete.")
                        print("Saved actual file:", local_file)
                        return local_file

                    elif current_size > remote_size:
                        print("Temporary file is larger than expected.")
                        print("Deleting temp file and retrying from scratch.")
                        temp_file.unlink()

                    else:
                        print("Download incomplete. Will retry/resume.")

                else:
                    # If server did not provide size, assume success if no exception.
                    temp_file.rename(local_file)
                    print("Download complete.")
                    print("Saved actual file:", local_file)
                    return local_file

        except Exception as e:
            print("Download error:")
            print(e)

        if attempt < MAX_RETRIES:
            print(f"Waiting {WAIT_SECONDS} seconds before retrying...")
            time.sleep(WAIT_SECONDS)

    raise RuntimeError(
        f"Could not download complete NOAA file after {MAX_RETRIES} attempts."
    )


# ------------------------------------------------------------
# Read actual temperature data
# ------------------------------------------------------------

actual_file = download_actual_year_file(YEAR)

print()
print("Opening local actual dataset...")
ds_actual = xr.open_dataset(actual_file)

# ------------------------------------------------------------
# Keep only requested date range
# ------------------------------------------------------------

ds_actual = ds_actual.sel(
    time=slice(START_DATE, END_DATE)
)

# ------------------------------------------------------------
# Keep only requested spatial area
#
# NCEP Reanalysis 2:
# lat usually runs north -> south
# lon is usually 0 to 360 degrees
#
# USA rough box:
# lat 50 to 25
# lon 235 to 295
#
# 235E = 125W
# 295E = 65W
# ------------------------------------------------------------

ds_actual = ds_actual.sel(
    lat=slice(50, 25),
    lon=slice(235, 295)
)

# ------------------------------------------------------------
# Convert xarray to dataframe
# ------------------------------------------------------------

actual_df = ds_actual["air"].to_dataframe().reset_index()

ds_actual.close()

# ------------------------------------------------------------
# Convert Kelvin to Celsius
# ------------------------------------------------------------

actual_df["actual_temp_2m_c"] = actual_df["air"] - 273.15

actual_df = actual_df[
    [
        "time",
        "lat",
        "lon",
        "actual_temp_2m_c"
    ]
]

actual_df["time"] = pd.to_datetime(actual_df["time"])

print()
print(actual_df.head())
print(actual_df.shape)

NOAA actual file URL:
https://psl.noaa.gov/thredds/fileServer/Datasets/ncep.reanalysis2/gaussian_grid/air.2m.gauss.2015.nc
Remote file size: 55,481,664 bytes
Using cached actual file: noaa_actual_cache\air.2m.gauss.2015.nc
Local file size: 55,481,664 bytes

Opening local actual dataset...

        time        lat      lon  actual_temp_2m_c
0 2015-01-01  48.570499  236.250         -4.529999
1 2015-01-01  48.570499  238.125        -18.229996
2 2015-01-01  48.570499  240.000        -21.759995
3 2015-01-01  48.570499  241.875        -11.139984
4 2015-01-01  48.570499  243.750        -10.509979
(607360, 4)


In [ ]:
# ============================================================
# MATCH ACTUAL TEMPERATURE TO EACH FORECAST ROW
#
# forecast_df and actual_df are on different grids.
# Therefore we do NOT merge by exact lat/lon.
# Instead, for each forecast row, we find the nearest actual
# temperature using:
#   forecast valid_time -> actual time
#   forecast latitude   -> actual lat
#   forecast longitude  -> actual lon
#
# final_df will keep the same number of rows as forecast_df.
# ============================================================

# Safety checks so errors are clearer
required_forecast_cols = {"valid_time", "latitude", "longitude", "predicted_temp_c"}
required_actual_cols = {"time", "lat", "lon", "actual_temp_2m_c"}

missing_forecast = required_forecast_cols - set(forecast_df.columns)
missing_actual = required_actual_cols - set(actual_df.columns)

if missing_forecast:
    raise ValueError(f"forecast_df is missing columns: {missing_forecast}")

if missing_actual:
    raise ValueError(f"actual_df is missing columns: {missing_actual}")

# Make copies so the original dataframes stay unchanged
final_df = forecast_df.copy()
actual_lookup_df = actual_df.copy()

# Make sure time columns are datetime
final_df["valid_time"] = pd.to_datetime(final_df["valid_time"])
actual_lookup_df["time"] = pd.to_datetime(actual_lookup_df["time"])

# Convert actual dataframe into an xarray grid for nearest-neighbour lookup
actual_grid = (
    actual_lookup_df
    .set_index(["time", "lat", "lon"])
    ["actual_temp_2m_c"]
    .to_xarray()
)

# Function used for each forecast row
def get_nearest_actual_temp(row):
    """
    Finds the nearest actual/reanalysis 2m temperature for one forecast row.
    """
    return float(
        actual_grid.sel(
            time=row["valid_time"],
            lat=row["latitude"],
            lon=row["longitude"],
            method="nearest"
        ).values
    )

# Add actual temperature to every forecast row
final_df["actual_temp_2m_c"] = final_df.apply(get_nearest_actual_temp, axis=1)

# Calculate model error
# Positive error means the forecast was too cold.
# Negative error means the forecast was too warm.
final_df["model_error_c"] = (
    final_df["actual_temp_2m_c"] - final_df["predicted_temp_c"]
)

print(final_df.head())
print(final_df.shape)

# Optional: save the matched dataset
final_df.to_csv("gefs_forecast_with_actual_temp.csv", index=False)
print("Saved: gefs_forecast_with_actual_temp.csv")



   year         run       time          valid_time  lead_hours  latitude  \
0  2015  2015010100 2015-01-01 2015-01-02 06:00:00          30      50.0   
1  2015  2015010100 2015-01-01 2015-01-02 06:00:00          30      50.0   
2  2015  2015010100 2015-01-01 2015-01-02 06:00:00          30      50.0   
3  2015  2015010100 2015-01-01 2015-01-02 06:00:00          30      50.0   
4  2015  2015010100 2015-01-01 2015-01-02 06:00:00          30      50.0   

   longitude  predicted_temp_c  precipitation  cloud_cover  actual_temp_2m_c  \
0      235.0          3.480926            0.0         90.0          0.839996   
1      236.0         -2.639069            0.0         95.0          0.839996   
2      237.0         -5.359070            0.0         89.0          0.839996   
3      238.0         -6.209076            0.0         81.0         -3.399994   
4      239.0         -6.709076            0.0         93.0         -3.399994   

   model_error_c  
0      -2.640930  
1       3.479065  
2    

In [ ]:
# ============================================================
# ADD COPERNICUS LAND COVER + 20-DEGREE LOCATION BLOCKS + ENCODING
#
# This cell starts from final_df, because final_df already contains:
#   - predicted_temp_c
#   - actual_temp_2m_c
#   - model_error_c
#
# It then creates:
#   - longitude_180
#   - land_code
#   - land_type
#   - land_use_type
#   - lat_block / lon_block / location_block
#   - encoded_df, ready for machine learning
# ============================================================



# ------------------------------------------------------------
# 0. Start from final_df, NOT the old forecast_df
# ------------------------------------------------------------
# final_df was created in the previous cell after matching actual
# temperature and calculating model_error_c.

if "final_df" not in globals():
    raise NameError("Run the previous cell first. final_df does not exist yet.")

forecast_df = final_df.copy()


# ------------------------------------------------------------
# 1. Path to your downloaded Copernicus GeoTIFF
# ------------------------------------------------------------

landcover_tif = Path(
    "PROBAV_LC100_global_v3.0.1_2015-base_Discrete-Classification-map_EPSG-4326.tif"
)

if not landcover_tif.exists():
    raise FileNotFoundError(
        f"Could not find {landcover_tif}\n"
        "Put the Copernicus .tif file in the same folder as this notebook."
    )


# ------------------------------------------------------------
# 2. Convert NOAA longitude from 0-360 to -180 to 180
# ------------------------------------------------------------
# NOAA example:
#   235 = -125
#   260 = -100
#   295 = -65
# Copernicus/rasterio expects normal longitude values: -180 to 180.

forecast_df["longitude_180"] = forecast_df["longitude"].apply(
    lambda x: x - 360 if x > 180 else x
)


# ------------------------------------------------------------
# 3. Sample Copernicus land-cover code at each lat/lon point
# ------------------------------------------------------------
# rasterio wants coordinates as:
#   (longitude, latitude)

coords = list(zip(
    forecast_df["longitude_180"],
    forecast_df["latitude"]
))

land_codes = []

with rasterio.open(landcover_tif) as src:
    for value in src.sample(coords):
        code = int(value[0])

        # Some rasters use 255 for missing/unknown.
        if code in [255]:
            land_codes.append(np.nan)
        else:
            land_codes.append(code)

forecast_df["land_code"] = land_codes


# ------------------------------------------------------------
# 4. Convert Copernicus numeric codes into land type names
# ------------------------------------------------------------

copernicus_land_classes = {
    0: "Unknown",
    20: "Shrubs",
    30: "Herbaceous vegetation",
    40: "Cultivated and managed vegetation or agriculture",
    50: "Urban or built up",
    60: "Bare or sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    100: "Moss and lichen",

    111: "Closed forest, evergreen needle leaf",
    112: "Closed forest, evergreen broad leaf",
    113: "Closed forest, deciduous needle leaf",
    114: "Closed forest, deciduous broad leaf",
    115: "Closed forest, mixed",
    116: "Closed forest, unknown",

    121: "Open forest, evergreen needle leaf",
    122: "Open forest, evergreen broad leaf",
    123: "Open forest, deciduous needle leaf",
    124: "Open forest, deciduous broad leaf",
    125: "Open forest, mixed",
    126: "Open forest, unknown",

    200: "Open sea"
}

forecast_df["land_type"] = forecast_df["land_code"].map(copernicus_land_classes)
forecast_df["land_type"] = forecast_df["land_type"].fillna("Unknown")


# ------------------------------------------------------------
# 5. Create broader land-use type groups
# ------------------------------------------------------------

def classify_land_use_type(land_type):
    if pd.isna(land_type):
        return "Unknown"

    land_type_lower = str(land_type).lower()

    if "urban" in land_type_lower or "built" in land_type_lower:
        return "Urban"

    elif "cultivated" in land_type_lower or "agriculture" in land_type_lower:
        return "Agriculture"

    elif "forest" in land_type_lower:
        return "Forest"

    elif "water" in land_type_lower or "sea" in land_type_lower:
        return "Water"

    elif "wetland" in land_type_lower:
        return "Wetland"

    elif "snow" in land_type_lower or "ice" in land_type_lower:
        return "Snow/Ice"

    elif (
        "shrub" in land_type_lower
        or "herbaceous" in land_type_lower
        or "moss" in land_type_lower
        or "lichen" in land_type_lower
    ):
        return "Natural vegetation"

    elif "bare" in land_type_lower or "sparse" in land_type_lower:
        return "Bare ground"

    elif "unknown" in land_type_lower:
        return "Unknown"

    else:
        return "Other"


forecast_df["land_use_type"] = forecast_df["land_type"].apply(classify_land_use_type)


# ------------------------------------------------------------
# 6. Create 20-degree latitude/longitude blocks
# ------------------------------------------------------------
# Latitude normally runs from -90 to 90.
# Adding 90 shifts it to 0 to 180 before dividing into 20-degree blocks.
#
# Longitude here uses NOAA's 0 to 360 format.
# If you prefer -180 to 180 blocks, use longitude_180 instead.

forecast_df["lat_block"] = np.floor((forecast_df["latitude"] + 90) / 20).astype(int)
forecast_df["lon_block"] = np.floor(forecast_df["longitude"] / 20).astype(int)

# Combined location zone. Example: lat block 7 and lon block 11 becomes "7_11".
forecast_df["location_block"] = (
    forecast_df["lat_block"].astype(str)
    + "_"
    + forecast_df["lon_block"].astype(str)
)


# ------------------------------------------------------------
# 7. Check the unencoded result first
# ------------------------------------------------------------

print("Unencoded dataframe preview:")
print(
    forecast_df[
        [
            "valid_time",
            "latitude",
            "longitude",
            "longitude_180",
            "predicted_temp_c",
            "actual_temp_2m_c",
            "model_error_c",
            "land_code",
            "land_type",
            "land_use_type",
            "lat_block",
            "lon_block",
            "location_block"
        ]
    ].head(20)
)

print("forecast_df shape:", forecast_df.shape)


# ------------------------------------------------------------
# 8. One-hot encode categorical columns
# ------------------------------------------------------------
# Keep forecast_df unencoded for checking.
# Create encoded_df for machine learning.
#
# We encode location_block instead of separately encoding lat_block
# and lon_block, because location_block already combines both.

categorical_cols = [
    "land_type",
    "land_use_type",
    "location_block"
]

encoded_df = pd.get_dummies(
    forecast_df,
    columns=categorical_cols,
    dtype=int
)

print("encoded_df shape:", encoded_df.shape)
print("Encoded columns added:")
print([c for c in encoded_df.columns if c.startswith("land_type_")][:10])
print([c for c in encoded_df.columns if c.startswith("land_use_type_")][:10])
print([c for c in encoded_df.columns if c.startswith("location_block_")][:10])

encoded_df.columns

Unencoded dataframe preview:
            valid_time  latitude  longitude  longitude_180  predicted_temp_c  \
0  2015-01-02 06:00:00      50.0      235.0         -125.0          3.480926   
1  2015-01-02 06:00:00      50.0      236.0         -124.0         -2.639069   
2  2015-01-02 06:00:00      50.0      237.0         -123.0         -5.359070   
3  2015-01-02 06:00:00      50.0      238.0         -122.0         -6.209076   
4  2015-01-02 06:00:00      50.0      239.0         -121.0         -6.709076   
5  2015-01-02 06:00:00      50.0      240.0         -120.0         -8.809082   
6  2015-01-02 06:00:00      50.0      241.0         -119.0         -9.349091   
7  2015-01-02 06:00:00      50.0      242.0         -118.0         -9.459076   
8  2015-01-02 06:00:00      50.0      243.0         -117.0         -9.759064   
9  2015-01-02 06:00:00      50.0      244.0         -116.0        -13.019073   
10 2015-01-02 06:00:00      50.0      245.0         -115.0        -14.179077   
11 2015-01-

Index(['year', 'run', 'time', 'valid_time', 'lead_hours', 'latitude',
       'longitude', 'predicted_temp_c', 'precipitation', 'cloud_cover',
       'actual_temp_2m_c', 'model_error_c', 'longitude_180', 'land_code',
       'lat_block', 'lon_block', 'land_type_Bare or sparse vegetation',
       'land_type_Closed forest, deciduous broad leaf',
       'land_type_Closed forest, evergreen needle leaf',
       'land_type_Closed forest, mixed', 'land_type_Closed forest, unknown',
       'land_type_Cultivated and managed vegetation or agriculture',
       'land_type_Herbaceous vegetation', 'land_type_Herbaceous wetland',
       'land_type_Open forest, deciduous broad leaf',
       'land_type_Open forest, evergreen needle leaf',
       'land_type_Open forest, unknown', 'land_type_Open sea',
       'land_type_Permanent water bodies', 'land_type_Shrubs',
       'land_type_Urban or built up', 'land_use_type_Agriculture',
       'land_use_type_Bare ground', 'land_use_type_Forest',
       'land_use_

In [ ]:
# ============================================================
# MODEL 1: Five-base-model stacking ensemble
# WITH BAYESIAN HYPERPARAMETER OPTIMISATION
#
# Target:
#   model_error_c = actual_temp_2m_c - predicted_temp_c
#
# Base models:
#   1. Ridge
#   2. Random Forest
#   3. XGBoost
#   4. LightGBM
#   5. GBDT
#
# Meta model / stacking layer:
#   Linear Regression
# ============================================================


# ------------------------------------------------------------
# 1. Start with encoded data
# ------------------------------------------------------------

df = encoded_df.copy()

print("Encoded data shape:", df.shape)
print("Columns:", list(df.columns))


# ------------------------------------------------------------
# 2. Remove rows with missing target
# ------------------------------------------------------------

df = df.dropna(subset=["model_error_c"])


# ------------------------------------------------------------
# 3. Define target
# ------------------------------------------------------------
# We are predicting forecast error:
#   model_error_c = actual_temp_2m_c - predicted_temp_c
#
# If model_error_c is positive, the original forecast was too cold.
# If model_error_c is negative, the original forecast was too warm.
# ------------------------------------------------------------

y = df["model_error_c"]


# ------------------------------------------------------------
# 4. Define input features
# ------------------------------------------------------------
# Do NOT include actual_temp_2m_c or model_error_c in X.
# That would leak the answer into the model.
# ------------------------------------------------------------

drop_cols = [
    "model_error_c",
    "actual_temp_2m_c",
    "valid_time",
    "forecast_start_time",
    "longitude_180",
    "land_code"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Keep only numeric / boolean columns.
# One-hot encoded columns are usually numeric or boolean.
X = X.select_dtypes(include=[np.number, "bool"])

print("X shape:", X.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# 5. Train / validation / test split
# ------------------------------------------------------------
# Paper split:
#   70% training
#   20% validation
#   10% test
#
# Training set:
#   used to tune and train the five base models.
#
# Validation set:
#   used to train the Linear Regression stacking model.
#
# Test set:
#   held back for final evaluation.
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1/3,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ------------------------------------------------------------
# 5b. Clean column names for XGBoost / LightGBM
# ------------------------------------------------------------

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

clean_cols = [
    re.sub(r'[^A-Za-z0-9_]+', '_', str(c))
    for c in X_train.columns
]

X_train.columns = clean_cols
X_val.columns = clean_cols
X_test.columns = clean_cols


# ------------------------------------------------------------
# 6. Metric function
# ------------------------------------------------------------

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    # Avoid divide-by-zero problems in MAPE.
    mask = actual != 0
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


def rmse_score(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))


# ------------------------------------------------------------
# 7. Bayesian hyperparameter optimisation settings
# ------------------------------------------------------------
# Optuna's TPE sampler is a Bayesian/sequential model-based optimiser.
# It tries promising parameter regions based on previous trial results,
# rather than blindly trying random combinations.
#
# Increase N_TRIALS_PER_MODEL for a better search.
# Start small first because XGBoost / LightGBM / RandomForest can be slow.
# ------------------------------------------------------------

N_TRIALS_PER_MODEL = 25
BAYES_RANDOM_STATE = 42

# Use a small hold-out split from the training set only for Bayesian tuning.
# This keeps the main validation set clean for training the stacker.
X_bayes_train, X_bayes_valid, y_bayes_train, y_bayes_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=BAYES_RANDOM_STATE
)

# Install Optuna automatically if it is missing.
# If you do not want notebook installs, set AUTO_INSTALL_OPTUNA = False.
AUTO_INSTALL_OPTUNA = True

try:
    import optuna
except ImportError:
    if not AUTO_INSTALL_OPTUNA:
        raise ImportError(
            "Optuna is needed for Bayesian hyperparameter optimisation. "
            "Install it with: pip install optuna"
        )
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


# ------------------------------------------------------------
# 8. Model factory
# ------------------------------------------------------------
# This function builds a model from a trial.
# Optuna chooses the hyperparameters, trains the model, then checks RMSE
# on X_bayes_valid.
# ------------------------------------------------------------

def build_model_from_trial(model_name, trial):
    if model_name == "ridge":
        return Ridge(
            alpha=trial.suggest_float("alpha", 1e-4, 100.0, log=True),
            random_state=BAYES_RANDOM_STATE
        )

    if model_name == "rf":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 600, step=100),
            max_depth=trial.suggest_int("max_depth", 3, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
            random_state=BAYES_RANDOM_STATE,
            n_jobs=-1
        )

    if model_name == "gbdt":
        return GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 600, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.20, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            random_state=BAYES_RANDOM_STATE
        )

    if model_name == "xgb":
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.20, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            objective="reg:squarederror",
            random_state=BAYES_RANDOM_STATE,
            n_jobs=-1
        )

    if model_name == "lgbm":
        return LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.20, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 127),
            max_depth=trial.suggest_int("max_depth", -1, 12),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            objective="regression",
            random_state=BAYES_RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        )

    raise ValueError(f"Unknown model name: {model_name}")


def tune_model(model_name):
    def objective(trial):
        model = build_model_from_trial(model_name, trial)
        model.fit(X_bayes_train, y_bayes_train)
        valid_pred = model.predict(X_bayes_valid)
        return rmse_score(y_bayes_valid, valid_pred)

    sampler = optuna.samplers.TPESampler(seed=BAYES_RANDOM_STATE)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=N_TRIALS_PER_MODEL, show_progress_bar=False)
    return study


# ------------------------------------------------------------
# 9. Run Bayesian optimisation for each base model
# ------------------------------------------------------------

model_names = ["ridge", "rf", "gbdt", "xgb", "lgbm"]

optuna_studies = {}
best_params_by_model = {}
bayes_tuning_results = []

for model_name in model_names:
    print(f"\nBayesian tuning: {model_name}")
    study = tune_model(model_name)
    optuna_studies[model_name] = study
    best_params_by_model[model_name] = study.best_params

    bayes_tuning_results.append({
        "Model": model_name,
        "Best validation RMSE": study.best_value,
        "Best parameters": study.best_params
    })

    print("  Best RMSE:", round(study.best_value, 5))
    print("  Best params:", study.best_params)

bayes_tuning_results_df = pd.DataFrame(bayes_tuning_results)
print("\nBayesian optimisation summary:")
print(bayes_tuning_results_df[["Model", "Best validation RMSE"]])


# ------------------------------------------------------------
# 10. Define final base models using the best parameters
# ------------------------------------------------------------
# These are now trained on the full training set, not only on the
# Bayesian tuning subset.
# ------------------------------------------------------------

base_models = {
    name: build_model_from_trial(name, optuna.trial.FixedTrial(best_params_by_model[name]))
    for name in model_names
}


# ------------------------------------------------------------
# 11. Train base models and collect their predictions
# ------------------------------------------------------------

trained_base_models = {}

base_train_predictions = pd.DataFrame(index=X_train.index)
base_val_predictions = pd.DataFrame(index=X_val.index)
base_test_predictions = pd.DataFrame(index=X_test.index)

base_model_results = []

for name, model in base_models.items():
    print("\nTraining final base model:", name)

    model.fit(X_train, y_train)
    trained_base_models[name] = model

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    base_train_predictions[f"{name}_pred"] = train_pred
    base_val_predictions[f"{name}_pred"] = val_pred
    base_test_predictions[f"{name}_pred"] = test_pred

    for dataset_name, y_split, preds in [
        ("Training Set", y_train, train_pred),
        ("Validation Set", y_val, val_pred),
        ("Test Set", y_test, test_pred)
    ]:
        base_model_results.append({
            "Model": name,
            "Dataset": dataset_name,
            **calculate_metrics(y_split, preds)
        })

base_results_df = pd.DataFrame(base_model_results)


# ------------------------------------------------------------
# 12. Train stacking model
# ------------------------------------------------------------
# The stacker does not use the original columns directly.
# It only sees the five base-model predictions:
#
#   ridge_pred, rf_pred, gbdt_pred, xgb_pred, lgbm_pred
# ------------------------------------------------------------

ensemble_features = list(base_val_predictions.columns)

X_stack_train = base_val_predictions[ensemble_features]
y_stack_train = y_val

stacking_model = LinearRegression()
stacking_model.fit(X_stack_train, y_stack_train)

print("\nStacking features:", ensemble_features)
print("Stacking coefficients:")
for feature, coef in zip(ensemble_features, stacking_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print("Stacking intercept:", stacking_model.intercept_)


# ------------------------------------------------------------
# 13. Final stacked predictions
# ------------------------------------------------------------

stack_train_pred = stacking_model.predict(base_train_predictions[ensemble_features])
stack_val_pred = stacking_model.predict(base_val_predictions[ensemble_features])
stack_test_pred = stacking_model.predict(base_test_predictions[ensemble_features])

stack_results_df = pd.DataFrame([
    {
        "Model": "Stacked_LinearRegression_BayesianTuned",
        "Dataset": "Training Set",
        **calculate_metrics(y_train, stack_train_pred)
    },
    {
        "Model": "Stacked_LinearRegression_BayesianTuned",
        "Dataset": "Validation Set",
        **calculate_metrics(y_val, stack_val_pred)
    },
    {
        "Model": "Stacked_LinearRegression_BayesianTuned",
        "Dataset": "Test Set",
        **calculate_metrics(y_test, stack_test_pred)
    }
])

results_df = pd.concat([base_results_df, stack_results_df], ignore_index=True)


# ------------------------------------------------------------
# 14. Store test-set error predictions for the next cell
# ------------------------------------------------------------

predicted_error_test = stack_test_pred

# Keep base model predictions too, so you can inspect the ensemble.
stack_test_details = base_test_predictions.copy()
stack_test_details["stacked_predicted_error_c"] = predicted_error_test
stack_test_details["actual_model_error_c"] = y_test.values

print("\nModel results:")
print(results_df.sort_values(["Dataset", "RMSE"]))

results_df


In [ ]:
# ============================================================
# Compare original forecast vs stacked corrected forecast
# ============================================================


# ------------------------------------------------------------
# 1. Get original rows for the held-out test set
# ------------------------------------------------------------

test_output = df.loc[X_test.index].copy()


# ------------------------------------------------------------
# 2. Store stacked predicted error
# ------------------------------------------------------------

test_output["predicted_error_c"] = predicted_error_test


# ------------------------------------------------------------
# 3. Correct the original forecast
# ------------------------------------------------------------
# model_error_c = actual_temp_2m_c - predicted_temp_c
# therefore:
# corrected_temp_c = predicted_temp_c + predicted_error_c
# ------------------------------------------------------------

test_output["corrected_temp_c"] = (
    test_output["predicted_temp_c"] + test_output["predicted_error_c"]
)


# ------------------------------------------------------------
# 4. Metric function
# ------------------------------------------------------------

def calculate_forecast_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    mask = actual != 0
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


# ------------------------------------------------------------
# 5. Original forecast vs corrected forecast
# ------------------------------------------------------------

original_metrics = calculate_forecast_metrics(
    test_output["actual_temp_2m_c"],
    test_output["predicted_temp_c"]
)

corrected_metrics = calculate_forecast_metrics(
    test_output["actual_temp_2m_c"],
    test_output["corrected_temp_c"]
)

comparison_df = pd.DataFrame([
    {
        "Forecast Type": "Original Forecast",
        **original_metrics
    },
    {
        "Forecast Type": "Stacked Corrected Forecast",
        **corrected_metrics
    }
])

print(comparison_df)

if corrected_metrics["RMSE"] < original_metrics["RMSE"]:
    print("Good: stacked correction improved the forecast RMSE.")
else:
    print("Warning: stacked correction did not improve the forecast RMSE yet.")


# ------------------------------------------------------------
# 6. Add individual base-model error predictions for inspection
# ------------------------------------------------------------

if "base_test_predictions" in globals():
    for col in base_test_predictions.columns:
        test_output[col] = base_test_predictions[col]


# ------------------------------------------------------------
# 7. Show sample results safely
# ------------------------------------------------------------

wanted_cols = [
    "latitude",
    "longitude",
    "lead_hours",
    "actual_temp_2m_c",
    "predicted_temp_c",
    "ridge_pred",
    "rf_pred",
    "xgb_pred",
    "lgbm_pred",
    "gbdt_pred",
    "predicted_error_c",
    "corrected_temp_c"
]

existing_cols = [c for c in wanted_cols if c in test_output.columns]

test_output[existing_cols].head(20)
